# Bases, variance and the metric

The one notion `TensND` has that a Cartesian-only tensor library does not: a
tensor carries a **basis**, which need not be orthonormal, and a **variance**
saying how each index transforms. This tutorial builds an oblique basis, moves
components between variances, and checks the identities of
Bases and variance — symbolically and numerically.

The definitions in one place: for a basis $(\underline{e}_i)$ the dual basis
$(\underline{e}^i)$ satisfies $\underline{e}^i\cdot\underline{e}_j=\delta^i_j$,
and the two metrics are

$$
g_{ij}=\underline{e}_i\cdot\underline{e}_j,
\qquad
g^{ij}=\underline{e}^i\cdot\underline{e}^j,
\qquad
g^{ik}g_{kj}=\delta^i_j .
$$

In [1]:
using TensND
using LinearAlgebra
using SymPy
using Tensors

## An oblique basis

The constructor takes the new basis vectors as **columns** expressed in the
canonical basis. Nothing here is orthogonal or normalized.

In [2]:
ℬ = Basis(Sym[1 0 0; 0 1 0; 0 1 1])

3×3 Basis{3, Sym}:
 1  0  0
 0  1  0
 0  1  1

The four stored matrices. `vecbasis(ℬ, :cov)` holds the $\underline{e}_i$,
`vecbasis(ℬ, :cont)` the dual vectors $\underline{e}^i$.

In [3]:
vecbasis(ℬ, :cov)

3×3 Matrix{Sym}:
 1  0  0
 0  1  0
 0  1  1

In [4]:
vecbasis(ℬ, :cont)

3×3 Matrix{Sym}:
 1  0   0
 0  1  -1
 0  0   1

The covariant metric $g_{ij}$ and its inverse $g^{ij}$:

In [5]:
metric(ℬ, :cov)

3×3 LinearAlgebra.Symmetric{Sym, Matrix{Sym}}:
 1  0  0
 0  2  1
 0  1  1

In [6]:
metric(ℬ, :cont)

3×3 LinearAlgebra.Symmetric{Sym, Matrix{Sym}}:
 1   0   0
 0   1  -1
 0  -1   2

## The defining identities

$g^{ik}g_{kj}=\delta^i_j$ and
$\underline{e}^i\cdot\underline{e}_j=\delta^i_j$:

In [7]:
tsimplify(metric(ℬ, :cont) * metric(ℬ, :cov))

3×3 Matrix{Sym{PyCall.PyObject}}:
 1  0  0
 0  1  0
 0  0  1

In [8]:
tsimplify(transpose(vecbasis(ℬ, :cont)) * vecbasis(ℬ, :cov))

3×3 Matrix{Sym{PyCall.PyObject}}:
 1  0  0
 0  1  0
 0  0  1

The basis is neither orthogonal nor orthonormal, which is exactly why variance
will matter below:

In [9]:
isorthogonal(ℬ), isorthonormal(ℬ)

(false, false)

## Covariant and contravariant components of a vector

The same vector, two sets of components. `TensND` stores a tensor together
with its basis and variance, and `components` converts.

In [10]:
V = Tens(Tensor{1, 3}(i -> symbols("v$i", real = true)))

3-element TensND.TensCanonical{1, 3, Sym{PyCall.PyObject}, Tensors.Vec{3, Sym{PyCall.PyObject}}}:
 v₁
 v₂
 v₃

Contravariant components $v^i$ (the coefficients on $\underline{e}_i$):

In [11]:
components(V, ℬ, (:cont,))

3-element Vector{Sym{PyCall.PyObject}}:
       v₁
       v₂
 -v₂ + v₃

Covariant components $v_i$ (the projections on $\underline{e}_i$):

In [12]:
components(V, ℬ, (:cov,))

3-element Vector{Sym{PyCall.PyObject}}:
      v₁
 v₂ + v₃
      v₃

They are related by the metric, $v_i = g_{ij}v^j$:

In [13]:
tsimplify(metric(ℬ, :cov) * components(V, ℬ, (:cont,)) - components(V, ℬ, (:cov,)))

3-element Vector{Sym{PyCall.PyObject}}:
 0
 0
 0

## Order 2: four variance combinations

An order-$p$ tensor has one choice per index, so an order-2 tensor has four.

In [14]:
T = Tens(Tensor{2, 3}((i, j) -> symbols("t$i$j", real = true)))

3×3 TensND.TensCanonical{2, 3, Sym{PyCall.PyObject}, Tensors.Tensor{2, 3, Sym{PyCall.PyObject}, 9}}:
 t₁₁  t₁₂  t₁₃
 t₂₁  t₂₂  t₂₃
 t₃₁  t₃₂  t₃₃

In [15]:
components(T, ℬ, (:cov, :cov))

3×3 Matrix{Sym{PyCall.PyObject}}:
       t₁₁              t₁₂ + t₁₃        t₁₃
 t₂₁ + t₃₁  t₂₂ + t₂₃ + t₃₂ + t₃₃  t₂₃ + t₃₃
       t₃₁              t₃₂ + t₃₃        t₃₃

In [16]:
tfactor(tsimplify(components(T, ℬ, (:cont, :cov))))

3×3 Matrix{Sym{PyCall.PyObject}}:
        t₁₁               t₁₂ + t₁₃         t₁₃
        t₂₁               t₂₂ + t₂₃         t₂₃
 -t₂₁ + t₃₁  -t₂₂ - t₂₃ + t₃₂ + t₃₃  -t₂₃ + t₃₃

## The cheapest sanity check on a basis

Store the covariant metric as a twice-covariant tensor and raise one index.
Since $g^{ik}g_{kj}=\delta^i_j$, the answer *must* be the identity — whatever
the basis.

In [17]:
G = Tens(metric(ℬ, :cov), ℬ, (:cov, :cov))
components(G, (:cont, :cov))

3×3 Matrix{Sym{PyCall.PyObject}}:
 1  0  0
 0  1  0
 0  0  1

## Normalization removes the scaling, not the obliquity

Dividing each vector by its norm gives unit vectors, so the metric acquires a
unit diagonal — but the off-diagonal terms, which measure the angles between
the vectors, survive. Variance still matters.

In [18]:
ℬ̄ = normalize(ℬ)
metric(ℬ̄, :cov)

3×3 LinearAlgebra.Symmetric{Sym, Matrix{Sym}}:
 1          0          0
 0          1  sqrt(2)/2
 0  sqrt(2)/2          1

In [19]:
isorthogonal(ℬ̄), isorthonormal(ℬ̄)

(false, false)

## Where variance becomes invisible

On an orthonormal basis $g_{ij}=g^{ij}=\delta_{ij}$, so the two sets of
components coincide and the distinction collapses. This is the case for
`CanonicalBasis` and `RotatedBasis`, and it is why tensors on
those bases carry no variance tuple at all.

In [20]:
θ, ϕ, ψ = symbols("θ ϕ ψ", real = true)
ℬʳ = Basis(θ, ϕ, ψ)
typeof(ℬʳ)

RotatedBasis{3, Sym{PyCall.PyObject}}

In [21]:
isorthonormal(ℬʳ), tsimplify(metric(ℬʳ, :cov))

(true, Sym{PyCall.PyObject}[1 0 0; 0 1 0; 0 0 1])

The same vector expressed on a rotated basis: covariant and contravariant
components are equal.

In [22]:
W = Tens(Tensor{1, 3}(i -> symbols("w$i", real = true)))
tsimplify(components(W, ℬʳ, (:cov,)) - components(W, ℬʳ, (:cont,)))

3-element Vector{Sym{PyCall.PyObject}}:
 0
 0
 0

## Numerically, on an orthogonal (scaled) basis

An `OrthogonalBasis` is an orthonormal basis with a scaling factor
$\chi_i$ per direction — the situation of the *natural* basis of a
curvilinear coordinate system, where the $\chi_i$ are the Lamé coefficients
(see Curvilinear differential calculus). Its metric is
diagonal, $g_{ij}=\mathrm{diag}(\chi_i^2)$.

In [23]:
ℬ₀ = Basis(0.3, 0.7, 0.0)          # orthonormal, rotated
χ = (2.0, 3.0, 0.5)                 # scaling factors
ℬχ = Basis(ℬ₀, χ)
typeof(ℬχ)

OrthogonalBasis{3, Float64}

In [24]:
round.(metric(ℬχ, :cov), digits = 12)

3×3 LinearAlgebra.Diagonal{Float64, Vector{Float64}}:
 4.0   ⋅    ⋅ 
  ⋅   9.0   ⋅ 
  ⋅    ⋅   0.25

Diagonal, with entries $\chi_i^2$:

In [25]:
round.(diag(metric(ℬχ, :cov)) .- [χ[i]^2 for i in 1:3], digits = 14)

3-element Vector{Float64}:
 0.0
 0.0
 0.0

and the dual vectors are $\underline{e}^i=\underline{e}_i/\chi_i^2$:

In [26]:
round.(
    vecbasis(ℬχ, :cont) .- vecbasis(ℬχ, :cov) * diagm([1 / χ[i]^2 for i in 1:3]),
    digits = 14,
)

3×3 Matrix{Float64}:
 0.0  -0.0  0.0
 0.0   0.0  0.0
 0.0   0.0  0.0

## Summary

| Basis type | metric | variance matters? |
|:--|:--|:--|
| `CanonicalBasis` | $\boldsymbol{1}$ | no |
| `RotatedBasis` | $\boldsymbol{1}$ | no |
| `OrthogonalBasis` | $\mathrm{diag}(\chi_i^2)$ | yes, diagonally |
| `Basis` | full | yes |

The constructor always returns the most specific type that applies, so the
cost of a component conversion is decided by the basis, not by the caller.

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*